In [1]:
!pip -q install sentence-transformers faiss-cpu

In [1]:
import requests
import re
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup, NavigableString

from datetime import date

from sentence_transformers import SentenceTransformer

import faiss
import pickle

from IPython.display import display

In [2]:
sources = {
    "EXT-01": {
        "category": "Extension",
        "title": "Extensions",
        "url": "https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions",
        "type": "webpage"
    },

    "SC-01": {
        "category": "Special Consideration",
        "title": "Special Consideration",
        "url": "https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/special-consideration",
        "type": "webpage"
    },

    "EAA-01": {
        "category": "Equitable Assessment Arrangements",
        "title": "Equitable Assessment Arrangements",
        "url": "https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/equitable-assessment-arrangements",
        "type": "webpage"
    },

    "EAA-02": {
        "category": "Equitable Assessment Arrangements",
        "title": "Equitable Learning FAQs",
        "url": "https://www.rmit.edu.au/students/support-services/equitable-learning/faqs",
        "type": "webpage"
    },

    "POL-01": {
        "category": "Policy",
        "title": "Assessment and Assessment Flexibility Policy",
        "url": "https://policies.rmit.edu.au/document/view.php?id=7",
        "type": "policy"
    }
}

print("Number of sources:", len(sources))

for source_id, source in sources.items():
    print(
        source_id,
        "→",
        source["title"]
    )

Number of sources: 5
EXT-01 → Extensions
SC-01 → Special Consideration
EAA-01 → Equitable Assessment Arrangements
EAA-02 → Equitable Learning FAQs
POL-01 → Assessment and Assessment Flexibility Policy


In [3]:
def get_webpage(url):

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/139.0 Safari/537.36"
        ),
        "Accept-Language": "en-AU,en;q=0.9"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response

In [4]:
response = get_webpage(
    sources["EXT-01"]["url"]
)

print("Status code:", response.status_code)
print("Downloaded characters:", len(response.text))

Status code: 200
Downloaded characters: 168642


In [5]:
def scrape_rmit_policy(source_id, source):

    response = get_webpage(source["url"])

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Remove irrelevant page elements
    for tag in soup.find_all([
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "header",
        "iframe",
        "form",
        "aside"
    ]):
        tag.decompose()

    records = []

    # --------------------------------------------------
    # Find policy headings
    # --------------------------------------------------

    headings = soup.find_all(
        ["h1", "h2", "h3", "h4"]
    )

    print(
        "Policy headings found:",
        len(headings)
    )

    for heading in headings:

        section_title = clean_text(
            heading.get_text(
                " ",
                strip=True
            )
        )

        if not section_title:
            continue

        content_parts = []

        # --------------------------------------------------
        # Walk through following elements
        # --------------------------------------------------

        for element in heading.find_all_next():

            # Stop at next heading of same/higher level
            if element.name in [
                "h1",
                "h2",
                "h3",
                "h4"
            ]:

                if element != heading:

                    # For policy sections, stop at any
                    # subsequent heading.
                    break

            # Paragraphs
            if element.name == "p":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # List items
            elif element.name == "li":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # Table cells
            elif element.name in [
                "td",
                "th"
            ]:

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

        # --------------------------------------------------
        # Combine content
        # --------------------------------------------------

        content = " ".join(
            content_parts
        )

        content = clean_text(
            content
        )

        # Ignore tiny sections
        if len(content) < 50:
            continue

        records.append({

            "source_id":
                source_id,

            "category":
                source["category"],

            "title":
                source["title"],

            "section":
                section_title,

            "content":
                content,

            "source_url":
                source["url"],

            "source_type":
                source["type"],

            "collection_date":
                str(date.today()),

            "effective_date":
                "2026-07-28"

        })

    return records

In [6]:
def clean_text(text):

    text = str(text)

    # Replace non-breaking spaces
    text = text.replace("\xa0", " ")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [7]:
def scrape_rmit_page(source_id, source):

    response = get_webpage(source["url"])

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # --------------------------------------------------
    # Remove elements that are never useful for the RAG
    # --------------------------------------------------

    for tag in soup.find_all([
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "header",
        "iframe",
        "form"
    ]):
        tag.decompose()

    # --------------------------------------------------
    # Find the page title
    # --------------------------------------------------

    h1 = soup.find("h1")

    if h1:
        page_title = clean_text(
            h1.get_text(" ", strip=True)
        )
    else:
        page_title = source["title"]

    # --------------------------------------------------
    # Find all headings
    # --------------------------------------------------

    headings = soup.find_all(
        ["h1", "h2", "h3"]
    )

    print(
        f"{source['title']} - headings found:",
        len(headings)
    )

    records = []

    # --------------------------------------------------
    # Extract sections
    # --------------------------------------------------

    for heading in headings:

        section_title = clean_text(
            heading.get_text(" ", strip=True)
        )

        if not section_title:
            continue

        content_parts = []

        # Walk through elements after heading
        # but stop at the next heading.
        for element in heading.find_all_next():

            # Stop when another heading is reached
            if element.name in ["h1", "h2", "h3"]:
                if element != heading:
                    break

            # Extract paragraphs
            if element.name == "p":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # Extract list items
            elif element.name == "li":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # Extract table cells
            elif element.name in ["td", "th"]:

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

        # --------------------------------------------------
        # Combine section content
        # --------------------------------------------------

        content = " ".join(content_parts)

        content = clean_text(content)

        # --------------------------------------------------
        # Remove navigation contamination
        # --------------------------------------------------

        navigation_phrases = [
            "RMIT Europe",
            "RMIT Global",
            "RMIT Vietnam",
            "RMIT UP",
            "RMIT Online",
            "New students",
            "Enrol as a new student",
            "Before semester starts",
            "New research students",
            "My course"
        ]

        # If many navigation phrases occur,
        # discard this record.
        navigation_matches = sum(
            phrase.lower() in content.lower()
            for phrase in navigation_phrases
        )

        if navigation_matches >= 2:
            continue

        # --------------------------------------------------
        # Skip extremely short sections
        # --------------------------------------------------

        if len(content) < 50:
            continue

        records.append({

            "source_id": source_id,

            "category": source["category"],

            "title": page_title,

            "section": section_title,

            "content": content,

            "source_url": source["url"],

            "source_type": source["type"],

            "collection_date": str(date.today())

        })

    return records

In [8]:
all_records = []

for source_id, source in sources.items():

    print("\n" + "=" * 70)
    print("Scraping:", source["title"])
    print("=" * 70)

    try:

        if source["type"] == "policy":

            records = scrape_rmit_policy(
                source_id,
                source
            )

        else:

            records = scrape_rmit_page(
                source_id,
                source
            )

        print(
            "Sections collected:",
            len(records)
        )

        all_records.extend(
            records
        )

    except Exception as e:

        print(
            "ERROR:",
            source["title"]
        )

        print(e)


print("\n" + "=" * 70)

print(
    "TOTAL SECTIONS:",
    len(all_records)
)

print("=" * 70)


Scraping: Extensions
Extensions - headings found: 7
Sections collected: 4

Scraping: Special Consideration
Special Consideration - headings found: 15
Sections collected: 8

Scraping: Equitable Assessment Arrangements
Equitable Assessment Arrangements - headings found: 8
Sections collected: 2

Scraping: Equitable Learning FAQs
Equitable Learning FAQs - headings found: 45
Sections collected: 34

Scraping: Assessment and Assessment Flexibility Policy
Policy headings found: 17
Sections collected: 16

TOTAL SECTIONS: 64


In [9]:
web_df = pd.DataFrame(all_records)

print("Rows:", len(web_df))
print("Columns:", list(web_df.columns))

Rows: 64
Columns: ['source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date']


In [10]:
print(
    web_df.groupby(
        [
            "source_id",
            "category"
        ]
    ).size()
)

source_id  category                         
EAA-01     Equitable Assessment Arrangements     2
EAA-02     Equitable Assessment Arrangements    34
EXT-01     Extension                             4
POL-01     Policy                               16
SC-01      Special Consideration                 8
dtype: int64


In [11]:
display(web_df.head(10))

,source_id,category,title,section,content,source_url,source_type,collection_date,effective_date
0,EXT-01,Extension,Extensions,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
1,EXT-01,Extension,Extensions,Assessments eligible for an extension,You can apply for an extension for assessments...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
2,EXT-01,Extension,Extensions,How to apply,You must apply at least one working day before...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
3,EXT-01,Extension,Extensions,False documents and misleading information,"Creating, submitting or using fraudulent docum...",https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
4,SC-01,Special Consideration,Special consideration,Special consideration,If unexpected circumstances beyond your contro...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
5,SC-01,Special Consideration,Special consideration,If unexpected circumstances beyond your contro...,What is special consideration? Eligibility How...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
6,SC-01,Special Consideration,Special consideration,What is special consideration?,If unexpected circumstances outside your contr...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
7,SC-01,Special Consideration,Special consideration,Eligibility,If unexpected circumstances outside your contr...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
8,SC-01,Special Consideration,Special consideration,How to apply,Most students can apply for special considerat...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN
9,SC-01,Special Consideration,Special consideration,Supporting documentation,"If you’re applying for special consideration, ...",https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,NaN


In [12]:
print(
    web_df.groupby(
        ["source_id", "category"]
    ).size()
)

source_id  category                         
EAA-01     Equitable Assessment Arrangements     2
EAA-02     Equitable Assessment Arrangements    34
EXT-01     Extension                             4
POL-01     Policy                               16
SC-01      Special Consideration                 8
dtype: int64


In [13]:
pd.set_option(
    "display.max_colwidth",
    300
)

display(
    web_df[
        [
            "source_id",
            "section",
            "content"
        ]
    ].head(20)
)

,source_id,section,content
0,EXT-01,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.","If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seve..."
1,EXT-01,Assessments eligible for an extension,"You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for spe..."
2,EXT-01,How to apply,"You must apply at least one working day before the assessment deadline . If you are applying for an extension on or after an assessment due date, you must apply for special consideration . If you have access to Canvas, use the Assessment Extensions Tool . For help using the tool, refer to the St..."
3,EXT-01,False documents and misleading information,"Creating, submitting or using fraudulent documents constitutes a breach of the RMIT Student Charter and may be deemed misconduct under the Student Conduct Policy. Providing RMIT with fraudulent documents is also a crime under the Victorian Crimes Act 1958. Students who submit fraudulent document..."
4,SC-01,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for spe..."
5,SC-01,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration.",What is special consideration? Eligibility How to apply Supporting documentation Technical issues during assessments
6,SC-01,What is special consideration?,"If unexpected circumstances outside your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. Special consideration is made available by the University on the understanding that students will use it sparingly and only in cases of p..."
7,SC-01,Eligibility,"If unexpected circumstances outside your control have affected your ability to complete an assessment, then you may be eligible to apply for special consideration. an unexpected short-term physical or mental health condition difficult personal circumstances or significant emotional disturbance u..."
8,SC-01,How to apply,Most students can apply for special consideration online (see exceptions below). Please ensure you include a personal statement and supporting documentation with your application. You must apply within five working days after the assessment date or due date . You can submit your application even...
9,SC-01,Supporting documentation,"If you’re applying for special consideration, it is essential that you provide relevant formal, independent and verifiable documentation to support your application. A personal statement alone is not sufficient to support an application for special consideration. You're required to provide forma..."


In [14]:
policy_df = web_df[
    web_df["source_id"] == "POL-01"
].copy()

print(
    "Policy records:",
    len(policy_df)
)

display(
    policy_df[
        [
            "section",
            "content"
        ]
    ]
)

Policy records: 16


,section,content
48,Assessment and Assessment Flexibility Policy,Section 1 - Purpose Section 2 - Scope Section 3 - Policy Design of Assessment Students and Assessment Academic Integrity Responsibilities in Assessment Assessment Flexibility Academic Progress Supplementary Assessment Processes Section 4 - Procedures and Resources Section 5 - Definitions
49,Section 1 - Purpose,"(1) To ensure: Relevant and authentic assessment that supports and enables students to demonstrate evidence of learning at the appropriate level of study. Flexible, equitable and inclusive assessment with a commitment to caring for students whose circumstances require assessment flexibility. Qua..."
50,Section 2 - Scope,"(2) This policy is made pursuant to the Asessment, Academic Progress and Appeals Regulations . Except where otherwise stated it applies to all courses and programs offered by RMIT in the following categories of award: Higher Degree by Research RMIT accredited programs Vocational Education Progra..."
51,Design of Assessment,"(4) Learning outcomes are a central focus for design, quality and standards in assessment. (5) All learning outcomes in a course are specified in the course guide and are assessed. (6) Assessment tasks are appropriately designed to measure student achievement of learning outcomes. (7) Assessment..."
52,Students and Assessment,"(15) Course guides specify all assessment requirements for a course and the weightings of each assessment task, if applicable. (16) Information provided to students on assessment tasks state the performance standards so that students understand the level of attainment required. (17) Students rec..."
53,Academic Integrity,(23) Students demonstrate academic integrity in their assessment practices by: engaging with assessment activities in an honest way providing accountability for the authorship and originality of work submitted acknowledging the work of others and the re-use of original work. (24) Staff take an e...
54,Responsibilities in Assessment,"(27) Deans/Heads of School/Cluster Directors are responsible for: applying university assessment policy and processes in their school, industry cluster or centre; convening course assessment committees and program assessment boards for each program in accordance with the membership and terms of ..."
55,Assessment Flexibility,"(33) The purpose of special consideration, extensions and individual assessment arrangements is to ensure that students experiencing extenuating circumstances or who have specific needs are appropriately supported and can seek assessment arrangements that provide the best circumstances for ongoi..."
56,Extensions,(34) Extensions are available for unforeseen circumstances of a short-term nature. (35) Applications are submitted to the school/industry cluster at least one working day before the due date for an assessment. (36) Extensions can be approved for up to one week (seven calendar days) after the due...
57,Special Consideration,"(41) Special consideration is available for unexpected circumstances outside students’ control. These include but are not limited to unexpected short-term ill health, and unavoidable family, work, cultural or religious commitments. (42) An application for special consideration is made in advance..."


In [15]:
clean_df = web_df.copy()

# Normalize whitespace
clean_df["content"] = (
    clean_df["content"]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

# Find duplicates
duplicate_mask = clean_df.duplicated(
    subset=["content"],
    keep=False
)

print(
    "Records involved in duplicate content:",
    duplicate_mask.sum()
)

Records involved in duplicate content: 0


In [16]:
display(
    clean_df.loc[
        duplicate_mask,
        [
            "source_id",
            "section",
            "content"
        ]
    ]
)

,source_id,section,content


In [17]:
raw_path = "./rmit_assessment_support_scraped_raw.csv"

web_df.to_csv(
    raw_path,
    index=False,
    encoding="utf-8-sig"
)

print("Raw dataset saved:")
print(raw_path)

Raw dataset saved:
./rmit_assessment_support_scraped_raw.csv


In [18]:
navigation_phrases = [
    "RMIT Global",
    "RMIT Vietnam",
    "RMIT UP",
    "RMIT Online",
    "New students",
    "Enrol as a new student",
    "Before semester starts",
    "New research students",
    "My course"
]

for phrase in navigation_phrases:

    count = web_df["content"].str.contains(
        phrase,
        case=False,
        na=False
    ).sum()

    print(
        f"{phrase} → {count}"
    )

RMIT Global → 0
RMIT Vietnam → 0
RMIT UP → 2
RMIT Online → 0
New students → 0
Enrol as a new student → 0
Before semester starts → 0
New research students → 0
My course → 0


In [19]:
duplicate_mask = web_df.duplicated(
    subset=["content"],
    keep=False
)

print(
    "Records involved in duplicate content:",
    duplicate_mask.sum()
)

Records involved in duplicate content: 0


In [20]:
display(
    web_df.loc[
        duplicate_mask,
        [
            "source_id",
            "section",
            "content"
        ]
    ]
)

,source_id,section,content


In [21]:
clean_df = web_df.copy()

# Normalize content
clean_df["content"] = (
    clean_df["content"]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

# Remove exact duplicate content
before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["content"],
    keep="first"
).reset_index(drop=True)

after = len(clean_df)

print(
    "Records before:",
    before
)

print(
    "Duplicate records removed:",
    before - after
)

print(
    "Records after:",
    after
)

Records before: 64
Duplicate records removed: 0
Records after: 64


In [22]:
def contains_navigation(text):

    text = str(text).lower()

    navigation_phrases = [
        "rmit europe",
        "rmit global",
        "rmit vietnam",
        "rmit up",
        "rmit online",
        "new students",
        "enrol as a new student",
        "before semester starts",
        "new research students"
    ]

    matches = sum(
        phrase in text
        for phrase in navigation_phrases
    )

    return matches >= 2

In [23]:
navigation_mask = clean_df["content"].apply(
    contains_navigation
)

print(
    "Navigation records found:",
    navigation_mask.sum()
)

Navigation records found: 0


In [24]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["content"],
    keep="first"
).reset_index(drop=True)

after = len(clean_df)

print(
    "Records before:",
    before
)

print(
    "Duplicates removed:",
    before - after
)

print(
    "Records after:",
    after
)

Records before: 64
Duplicates removed: 0
Records after: 64


In [25]:
clean_df = clean_df[
    ~navigation_mask
].reset_index(drop=True)

print(
    "Final clean records:",
    len(clean_df)
)

Final clean records: 64


In [26]:
print(
    clean_df.isnull().sum()
)

source_id           0
category            0
title               0
section             0
content             0
source_url          0
source_type         0
collection_date     0
effective_date     48
dtype: int64


In [27]:
clean_df["content_length"] = (
    clean_df["content"]
    .str.len()
)

print(
    clean_df["content_length"].describe()
)

count       64.000000
mean      1201.250000
std       1776.174702
min        101.000000
25%        315.750000
50%        591.500000
75%       1284.250000
max      12073.000000
Name: content_length, dtype: float64


In [28]:
display(
    clean_df[
        clean_df["content_length"] < 100
    ][
        [
            "source_id",
            "section",
            "content",
            "content_length"
        ]
    ]
)

,source_id,section,content,content_length


In [29]:
clean_df["document_version"] = np.where(
    clean_df["source_id"] == "POL-01",
    "Current - effective 28 July 2026",
    "Current webpage"
)

In [40]:
print("\nMissing values:")
print(clean_df.isnull().sum())


Missing values:
source_id           0
category            0
title               0
section             0
content             0
source_url          0
source_type         0
collection_date     0
effective_date      0
content_length      0
document_version    0
dtype: int64


In [39]:
clean_df["effective_date"] = clean_df["effective_date"].fillna(
    "Not specified"
)

print(clean_df["effective_date"].value_counts())

effective_date
Not specified    48
2026-07-28       16
Name: count, dtype: int64


In [41]:
clean_path = (
    "./"
    "rmit_assessment_support_scraped_cleaned.csv"
)

clean_df.to_csv(
    clean_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Clean CSV saved:"
)

print(clean_path)

Clean CSV saved:
./rmit_assessment_support_scraped_cleaned.csv


In [42]:
print(
    "FINAL DATASET"
)

print(
    "Rows:",
    len(clean_df)
)

print(
    "Columns:",
    list(clean_df.columns)
)

display(
    clean_df[
        [
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

FINAL DATASET
Rows: 64
Columns: ['source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date', 'content_length', 'document_version']


,source_id,category,section,content
0,EXT-01,Extension,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.","If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seve..."
1,EXT-01,Extension,Assessments eligible for an extension,"You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for spe..."
2,EXT-01,Extension,How to apply,"You must apply at least one working day before the assessment deadline . If you are applying for an extension on or after an assessment due date, you must apply for special consideration . If you have access to Canvas, use the Assessment Extensions Tool . For help using the tool, refer to the St..."
3,EXT-01,Extension,False documents and misleading information,"Creating, submitting or using fraudulent documents constitutes a breach of the RMIT Student Charter and may be deemed misconduct under the Student Conduct Policy. Providing RMIT with fraudulent documents is also a crime under the Victorian Crimes Act 1958. Students who submit fraudulent document..."
4,SC-01,Special Consideration,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for spe..."
...,...,...,...,...
59,POL-01,Policy,Academic Progress,(67) Chairs of program assessment boards are responsible for: reviewing the progress of students in accordance with the academic progress (coursework programs) section of the assessment processes; determining the stage of unsatisfactory progress of students: First stage – at risk of unsatisfacto...
60,POL-01,Policy,Supplementary Assessment,(75) Supplementary assessment is approved by: course assessment committees for courses in programs the school/industry cluster delivers Program Managers for courses delivered by another school/industry cluster program assessment boards where a student has narrowly failed one course in the last t...
61,POL-01,Policy,Processes,"(78) The University Secretary and Academic Registrar: maintains the assessment processes for RMIT University, and approves the annual Academic Calendar. (79) The Chief Executive Officer, RMIT UP maintains the assessment processes for RMIT UP. (80) The Deputy Vice-Chancellor Vocational Education ..."
62,POL-01,Policy,Section 4 - Procedures and Resources,(81) Refer to the following documents which are established in accordance with this policy: Assessment and Assessment Flexibility – Online Invigilated Examination Procedure Assessment Processes Student Administration Resources Short assessment extension of time process


In [33]:
web_df["effective_date"] = web_df["effective_date"].fillna(
    "Not specified"
)

print(web_df["effective_date"].value_counts())

effective_date
Not specified    48
2026-07-28       16
Name: count, dtype: int64


In [36]:
web_df["effective_date"] = web_df["effective_date"].fillna(
    "Not specified"
)

print(web_df["effective_date"].value_counts())

effective_date
Not specified    48
2026-07-28       16
Name: count, dtype: int64


In [37]:
print("\nMissing values:")
print(web_df.isnull().sum())


Missing values:
source_id          0
category           0
title              0
section            0
content            0
source_url         0
source_type        0
collection_date    0
effective_date     0
dtype: int64
